In [2]:
import numpy as np
import os
from PIL import Image, ImageOps
from IPython.display import display


In [3]:
# File paths to different assets
file_enemy = "enemy"
file_env = "environment"

In [ ]:
def load_environment_images(directory_path):
    # Load all PNG images from the specified directory
    for file_name in os.listdir(directory_path):
        if file_name.endswith('.png'):
            img = Image.open(os.path.join(directory_path, file_name))
            # Resize the image to 256x256
            img_resized = img.resize((256, 256))
            # Display the resized image
            display(img_resized)

load_environment_images(file_env)

In [ ]:
def augment_image(img):
    # Flip the image horizontally
    img_flipped = ImageOps.mirror(img)
    
    # Mirror left half to right
    left_half = img.crop((0, 0, img.width // 2, img.height))
    left_mirrored = ImageOps.mirror(left_half)
    img_left_mirror = Image.new('RGB', (img.width, img.height))
    img_left_mirror.paste(left_half, (0, 0))
    img_left_mirror.paste(left_mirrored, (img.width // 2, 0))
    
    # Mirror right half to left
    right_half = img.crop((img.width // 2, 0, img.width, img.height))
    right_mirrored = ImageOps.mirror(right_half)
    img_right_mirror = Image.new('RGB', (img.width, img.height))
    img_right_mirror.paste(right_mirrored, (0, 0))
    img_right_mirror.paste(right_half, (img.width // 2, 0))
    
    # Copy 1/4 of the image to create 4 new images
    quarter_width = img.width // 2
    quarter_height = img.height // 2
    quarters = [
        img.crop((0, 0, quarter_width, quarter_height)),
        img.crop((quarter_width, 0, img.width, quarter_height)),
        img.crop((0, quarter_height, quarter_width, img.height)),
        img.crop((quarter_width, quarter_height, img.width, img.height))
    ]
    img_quarters = []
    for quarter in quarters:
        new_img = Image.new('RGB', (img.width, img.height))
        new_img.paste(quarter, (0, 0))
        new_img.paste(quarter, (quarter_width, 0))
        new_img.paste(quarter, (0, quarter_height))
        new_img.paste(quarter, (quarter_width, quarter_height))
        img_quarters.append(new_img)
    
    return [img_flipped, img_left_mirror, img_right_mirror] + img_quarters

In [5]:
def load_and_augment_images(directory_path, output_directory):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    # Load all PNG images from the specified directory
    for file_name in os.listdir(directory_path):
        if file_name.endswith('.png'):
            img = Image.open(os.path.join(directory_path, file_name))
            # Resize the image to 640x640
            img_resized = img.resize((640, 640))
            # Display the original image
            #display(img_resized)
            # Perform augmentations
            augmented_images = augment_image(img_resized)
            # Save the original and augmented images
            base_name = os.path.splitext(file_name)[0]
            img_resized.save(os.path.join(output_directory, f"{base_name}_original.png"))
            for i, aug_img in enumerate(augmented_images):
                aug_img.save(os.path.join(output_directory, f"{base_name}_aug_{i}.png"))
            #break  # Remove this line to process all images in the directory


In [ ]:
load_and_augment_images(file_env, 'env_processed')

In [5]:
import cv2
import time
import numpy as np
from PIL import Image
from ultralytics import YOLO

def preprocess_frame(frame):
    # Convert frame from BGR (OpenCV default) to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    
    # First crop: according to processing code (assumed frame is 2560x1440)
    crop_box_1 = (0, 0, 2080, 1440)
    image_cropped = pil_image.crop(crop_box_1)
    
    # Second crop: crop equally from left and right to obtain a 1440x1440 square
    crop_box_2 = (320, 0, 320 + 1440, 1440)
    final_image = image_cropped.crop(crop_box_2)
    
    # Resize to 640x640 (model input size)
    img_resized = final_image.resize((640, 640))
    
    # Convert back to BGR numpy array for OpenCV
    processed_frame = cv2.cvtColor(np.array(img_resized), cv2.COLOR_RGB2BGR)
    return processed_frame

# Load your trained model with weights
model = YOLO("../runs/detect/train5/weights/best.pt")

# Specify the path to your video file (or 0 for webcam)
video_path = r"D:\Vids\Desktop\Desktop 2025.02.10 - 10.43.42.23.DVR.mp4"  # Update as needed
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error opening video stream or file")
    exit()

frame_total = 0
frame_count = 0
total_inference_time = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_total += 1
    # Process only every second frame for performance.
    if frame_total % 10 != 0:
        continue

    processed_frame = preprocess_frame(frame)
    
    start_time = time.time()
    # Set confidence threshold to 0.6 to filter out weaker detections.
    results = model(processed_frame, conf=0.6)
    end_time = time.time()

    inference_time = end_time - start_time
    total_inference_time += inference_time
    frame_count += 1

    # Annotate the frame with detections (draw bounding boxes)
    annotated_frame = results[0].plot()  # returns an image with drawn boxes

    # Calculate instantaneous FPS and annotate
    fps = 1 / inference_time if inference_time > 0 else 0
    cv2.putText(annotated_frame, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

if frame_count > 0:
    avg_fps = frame_count / total_inference_time
    print(f"Processed {frame_count} frames. Average FPS: {avg_fps:.2f}")


0: 640x640 (no detections), 52.1ms
Speed: 1.0ms preprocess, 52.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 53.4ms
Speed: 1.5ms preprocess, 53.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 54.9ms
Speed: 1.0ms preprocess, 54.9ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 51.8ms
Speed: 1.0ms preprocess, 51.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 55.0ms
Speed: 1.5ms preprocess, 55.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 57.1ms
Speed: 1.5ms preprocess, 57.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 49.4ms
Speed: 1.5ms preprocess, 49.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 50.7ms
Speed: 1.0ms preprocess, 50.7ms i

In [ ]:
load_and_augment_images('whole_images_cropped', 'whole_images_processed')

In [ ]:
def place_enemy_on_environment(env_img, enemy_img, position):
    # Place the enemy image on the environment image at the specified position
    env_img.paste(enemy_img, position, enemy_img)
    return env_img

def load_and_place_enemy(env_directory, enemy_directory):
    # Load the first environment image
    env_file_name = next((f for f in os.listdir(env_directory) if f.endswith('.png')), None)
    if env_file_name is None:
        print("No environment images found.")
        return
    env_img = Image.open(os.path.join(env_directory, env_file_name))
    
    # Load the first enemy image
    enemy_file_name = next((f for f in os.listdir(enemy_directory) if f.endswith('.png')), None)
    if enemy_file_name is None:
        print("No enemy images found.")
        return
    enemy_img = Image.open(os.path.join(enemy_directory, enemy_file_name))
    
    # Resize the enemy image to twice its original size
    enemy_img_resized = enemy_img.resize((int(enemy_img.width * 4), int(enemy_img.height * 4)))
    
    # Place the enemy image on the environment image
    position = (env_img.width // 2 - enemy_img_resized.width // 2, env_img.height // 2 - enemy_img_resized.height // 2)
    result_img = place_enemy_on_environment(env_img, enemy_img_resized, position)
    
    # Display the resulting image
    display(result_img)

# Replace 'env_processed' with the actual path to your environment images directory
# Replace 'file_enemy' with the actual path to your enemy images directory
load_and_place_enemy('whole_images_cropped', file_enemy)

In [4]:
from PIL import Image
from IPython.display import display
import os
import random
import PIL.ImageChops as ImageChops
from PIL import ImageFilter
import cv2

def apply_filter_2(image, param1):
    """
    Applies a Sobel edge detector.
    'param1' is used as the kernel size.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Compute gradients along the x and y axis, using kernel size (must be odd)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=param1)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=param1)
    sobel = cv2.magnitude(sobelx, sobely)
    # Normalize the result to 0-255 then convert back to uint8
    sobel = np.uint8(255 * sobel / np.max(sobel))
    return cv2.cvtColor(sobel, cv2.COLOR_GRAY2BGR)

def add_outline(img, thickness=1, outline_color=(0, 0, 0, 255)):
    """
    Adds a crisp black outline to a transparent image by dilating the alpha channel.
    Uses an expansion approach to ensure the outline sits around the enemy.

    Parameters:
      img: PIL.Image object in RGBA mode.
      thickness: outline thickness in pixels.
      outline_color: RGBA tuple for the outline color.

    Returns:
      A new PIL.Image with an outline added.
    """
    if thickness <= 0:
        return img

    # Ensure the image is in RGBA.
    img = img.convert("RGBA")

    # Create a new blank image to accommodate the outline
    new_size = (img.width + thickness * 2, img.height + thickness * 2)
    canvas = Image.new("RGBA", new_size, (0, 0, 0, 0))
    canvas.paste(img, (thickness, thickness))

    # Extract and dilate the alpha channel
    alpha = canvas.split()[-1]
    dilated = alpha.filter(ImageFilter.MaxFilter(thickness * 2 + 1))

    # The outline mask is the dilated alpha minus the original alpha
    outline_mask = ImageChops.subtract(dilated, alpha)

    # Create an image for the outline using the outline_color
    outline_img = Image.new("RGBA", new_size, (0, 0, 0, 0))
    outline_img.paste(outline_color, mask=outline_mask)

    # Composite the outline and the original image
    final_img = Image.alpha_composite(outline_img, canvas)
    return final_img


def boxes_overlap(box1, box2):
    return not (box1[2] <= box2[0] or box1[0] >= box2[2] or 
                box1[3] <= box2[1] or box1[1] >= box2[3])

def is_mostly_transparent(img, threshold=0.5):
    if img.mode != "RGBA":
        img = img.convert("RGBA")
    alpha = img.getchannel("A")
    pixels = list(alpha.getdata())
    transparent_count = sum(1 for px in pixels if px == 0)
    return (transparent_count / len(pixels)) > threshold

def crop_enemy(enemy_img):
    if random.random() < 0.3:  # 30% chance to crop
        direction = random.choice(["left", "right", "top", "bottom"])
        crop_ratio = random.uniform(0, 0.3)  # Crop between 0-30%
        width, height = enemy_img.size

        if direction == "left":
            enemy_img = enemy_img.crop((int(width * crop_ratio), 0, width, height))
        elif direction == "right":
            enemy_img = enemy_img.crop((0, 0, int(width * (1 - crop_ratio)), height))
        elif direction == "top":
            enemy_img = enemy_img.crop((0, int(height * crop_ratio), width, height))
        elif direction == "bottom":
            enemy_img = enemy_img.crop((0, 0, width, int(height * (1 - crop_ratio))))
    return enemy_img

def generate_yolo_dataset(env_directory, enemy_directory, output_directory, val_split=0.2):
    # Create training and validation directories if they do not exist
    train_dir = os.path.join(output_directory, "training")
    val_dir = os.path.join(output_directory, "validation")
    for dir_path in (train_dir, val_dir):
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    enemy_files = [f for f in os.listdir(enemy_directory) if f.endswith('.png')]
    if not enemy_files:
        print("No enemy images found.")
        return

    restricted_areas = [
        (0, 0, 76, 144),    # Top-left corner 76x144
        (300, 287, 336, 326)  # Example restricted area
    ]

    for env_file in os.listdir(env_directory):
        if not env_file.endswith('.png'):
            continue
        
        env_path = os.path.join(env_directory, env_file)
        env_img = Image.open(env_path).convert("RGBA")
        env_width, env_height = env_img.size

        annotations = []
        placed_boxes = []
        n_enemies = random.randint(1, 4)
        enemy_count = 0

        while enemy_count < n_enemies:
            enemy_file = random.choice(enemy_files)
            enemy_path = os.path.join(enemy_directory, enemy_file)
            enemy_img = Image.open(enemy_path).convert("RGBA")
            
            if is_mostly_transparent(enemy_img, 0.5):
                continue

            enemy_img = crop_enemy(enemy_img)  # Apply random cropping
            
            # Resize the enemy image 4x its original dimensions
            enemy_img_resized = enemy_img.resize(
                (enemy_img.width * 2, enemy_img.height * 2), resample=Image.BICUBIC)
            enemy_img_resized = add_outline(enemy_img_resized, thickness=1)  # Add a 1-pixel black outline  # Add a 1-pixel black outline
            enemy_img_resized = enemy_img_resized.resize(
                (enemy_img_resized.width * 2, enemy_img_resized.height * 2), resample=Image.BOX)
            e_width, e_height = enemy_img_resized.size

            max_x = env_width - e_width
            max_y = env_height - e_height
            if max_x <= 0 or max_y <= 0:
                continue

            rand_x = random.randint(0, max_x)
            rand_y = random.randint(0, max_y)
            new_box = (rand_x, rand_y, rand_x + e_width, rand_y + e_height)

            if any(boxes_overlap(new_box, placed) for placed in placed_boxes):
                continue

            if any(boxes_overlap(new_box, restricted) for restricted in restricted_areas):
                continue

            env_img.paste(enemy_img_resized, (rand_x, rand_y), enemy_img_resized)
            placed_boxes.append(new_box)

            x_center = (rand_x + e_width / 2) / env_width
            y_center = (rand_y + e_height / 2) / env_height
            norm_width = e_width / env_width
            norm_height = e_height / env_height
            annotations.append(f"0 {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")
            enemy_count += 1

        # Determine whether to put in validation or training dataset
        if random.random() < val_split:
            out_dir = val_dir
        else:
            out_dir = train_dir

        output_img_path = os.path.join(out_dir, env_file)

        # Convert the PIL image (env_img) to a cv2 image for filtering:
        env_img_rgb = env_img.convert("RGB")
        env_np = np.array(env_img_rgb)
        env_cv = cv2.cvtColor(env_np, cv2.COLOR_RGB2BGR)

        # Apply the Sobel edge filter from filters.py with param1=3
        filtered_env_cv = apply_filter_2(env_cv, param1=3)

        # Convert back to a PIL image
        filtered_env_rgb = cv2.cvtColor(filtered_env_cv, cv2.COLOR_BGR2RGB)
        filtered_env_pil = Image.fromarray(filtered_env_rgb)

        # Save the filtered image for YOLO
        filtered_env_pil.save(output_img_path)

        annotation_path = output_img_path.replace('.png', '.txt')
        with open(annotation_path, 'w') as f:
            f.write("\n".join(annotations))

        print(f"Saved {env_file} and annotation to {out_dir}")

# Example usage:
# Replace 'whole_images_cropped' with your environment images directory.
# Replace 'file_enemy' with your enemy images directory.
# Replace 'dataset_yolo_whole' with your desired output directory.
generate_yolo_dataset('whole_images_cropped', file_enemy, 'dataset_yolo_whole')

Saved processed_Desktop Screenshot 2025.02.17 - 08.51.01.89.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.16.46.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.19.75.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.23.87.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.40.47.png and annotation to dataset_yolo_whole\validation
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.44.04.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.46.50.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.48.77.png and annotation to dataset_yolo_whole\training
Saved processed_Desktop Screenshot 2025.02.17 - 08.51.59.78.png and annotation to dataset_yolo_whole\training
Saved pr

In [1]:
import cv2
import time
from ultralytics import YOLO

# Load your trained model
model = YOLO("yolo11n.pt")

# Specify the path to your video file (or set to 0 for webcam)
video_path = "D:\Vids\Desktop\Desktop 2025.02.10 - 10.43.42.23.DVR.mp4"  # Change to your video file or use 0 for webcam

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error opening video stream or file")
    exit()

frame_count = 0
total_inference_time = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    start_time = time.time()
    results = model(frame)  # perform detection on the frame
    end_time = time.time()

    inference_time = end_time - start_time
    total_inference_time += inference_time
    frame_count += 1

    # Annotate the frame with detections
    annotated_frame = results[0].plot()  # returns an image with drawn boxes

    # Calculate instantaneous FPS and annotate
    fps = 1 / inference_time if inference_time > 0 else 0
    cv2.putText(annotated_frame, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

if frame_count > 0:
    avg_fps = frame_count / total_inference_time
    print(f"Processed {frame_count} frames. Average FPS: {avg_fps:.2f}")


0: 384x640 (no detections), 92.6ms
Speed: 5.0ms preprocess, 92.6ms inference, 3.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 49.2ms
Speed: 2.0ms preprocess, 49.2ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 42.0ms
Speed: 2.0ms preprocess, 42.0ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 62.0ms
Speed: 2.0ms preprocess, 62.0ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 51.2ms
Speed: 2.5ms preprocess, 51.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 56.5ms
Speed: 2.1ms preprocess, 56.5ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 bird, 43.7ms
Speed: 1.6ms preprocess, 43.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 bird, 54.3ms
Speed: 45.6ms preprocess, 54.3ms inference, 0.0ms p